# Ordinary Least Squares Experiment

This notebook explores the ordinary least squares (OLS) solution for Runge function. We evaluate the solution using the mean squared error (MSE) and R2 function. 

First we import all that is necesarry.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from fys_stk4155_p1.data.design_matrix import univariate_polynomial_design_matrix
from fys_stk4155_p1.data.runge import generate_runge_data, runge_function
from fys_stk4155_p1.metrics import mean_squared_error
from fys_stk4155_p1.regression.degree_sweep import fit_polynomial_degree_sweep
from fys_stk4155_p1.regression.ordinary_least_squares import OLS

We can then generate and visualize the data.

In [ ]:
# Generate data
x, y = generate_runge_data(n=100, noise_std=0.1, seed=42)


x_plot = np.linspace(-1, 1, 500)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x_plot, runge_function(x_plot), color="black", lw=1.5, label="Runge's function")
ax.scatter(x, y, s=15, alpha=0.6, label=r"data, $\sigma = 0.1$")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Runge function: noisy samples vs. ground truth")
ax.legend()
fig.tight_layout()
plt.show()

## Fit OLS across polynomial degree

For each degree $d$ we build the design matrix $[x, x^2, \dots, x^d]$ — no intercept
column (`intercept=False`), since `OLS`/`LinearModel` never adds one implicitly; see
`fys_stk4155_p1.regression.base`. To handle the intercept term $\theta_0$ without an
intercept column, and to avoid a `StandardScaler` dividing a constant column by a
standard deviation of zero, we follow the same convention already used by
`Ridge`/`shrinkage.py` elsewhere in this package (standardized features, centered
target, `fit_intercept_column=False`):

1. Fit a `StandardScaler` on the training split's columns and apply it to both train
   and test splits (fitting on train only avoids leaking test-set statistics into the
   transform).
2. Center `y` by subtracting the training mean, `y_mean = y_train.mean()`, and fit
   OLS on `(X_train_scaled, y_train - y_mean)`.
3. Because standardized features have zero mean, the true intercept is exactly
   `y_mean` — recovered by adding it back at prediction time
   (`y_pred = X_scaled @ theta + y_mean`) rather than by fitting it as an extra
   column.

With this layout `theta[i]` corresponds to the standardized $x^{i+1}$ term
(`theta[0]` is the $x^1$ coefficient, not an intercept), and the intercept is
returned separately.

> **TODO:** Part (a) also asks for a critical discussion of *why and how* we scaled
> the data. Write that discussion here once the design choices above are finalized
> (see Section 3.13 of the lecture notes, the Tuesday session of week 35 and the 
> pertinent notebook for week 35).

This design-matrix → scale → OLS → metrics logic is model-agnostic (it also drives
the upcoming Ridge experiment), so it lives in
`fys_stk4155_p1.regression.degree_sweep.fit_polynomial_degree_sweep` rather than as a
notebook-local helper — see that function's docstring for the full contract.

In [ ]:
# Baseline experiment: sweep polynomial degree at the original sample size (n=100).
degrees = range(1, 16)
results = fit_polynomial_degree_sweep(x, y, degrees, OLS)

degrees_arr = results["degrees"]
intercept = results["intercept"]
weights = results["weights"]
mse_train, mse_test = results["mse_train"], results["mse_test"]
r2_train, r2_test = results["r2_train"], results["r2_test"]

## MSE and R2 vs. polynomial degree

Plot train/test MSE and R2 side by side to see the bias-variance trade-off as model complexity grows.

In [ ]:
degrees_arr = np.array(list(degrees))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(degrees_arr, mse_train, marker="o", markersize=3, label="train")
axes[0].plot(degrees_arr, mse_test, marker="o", markersize=3, label="test")
axes[0].set_yscale("log")
axes[0].set_xlabel("Polynomial degree")
axes[0].set_ylabel("MSE")
axes[0].set_title("MSE vs. polynomial degree")
axes[0].legend()

axes[1].plot(degrees_arr, r2_train, marker="o", markersize=3, label="train")
axes[1].plot(degrees_arr, r2_test, marker="o", markersize=3, label="test")
axes[1].axhline(0, color="gray", lw=0.5)
axes[1].set_xlabel("Polynomial degree")
axes[1].set_ylabel(r"$R^2$")
axes[1].set_title(r"$R^2$ vs. polynomial degree")
axes[1].legend()

fig.tight_layout()
plt.show()

## Coefficients ($\theta$) vs. polynomial degree

Each fit has `degree` coefficients: `theta[i]` is the coefficient of the
standardized $x^{i+1}$ term (there's no intercept column anymore — the intercept is
`y_mean`, a single scalar shared across all degrees, shown separately). Since
`weights` is a ragged list (more coefficients at higher degrees), pad it into a
rectangular matrix (missing entries as `NaN`) so each coefficient $\theta_i$ can be
traced as its own line across the degrees where it exists. This shows the intercept
staying constant by construction (it never depends on degree) while the high-order
coefficients blow up as the fit starts overfitting.

In [ ]:
max_degree = max(degrees)
theta_matrix = np.full((len(degrees), max_degree), np.nan)
for row, theta in enumerate(weights):
    theta_matrix[row, : len(theta)] = theta

fig, ax = plt.subplots(figsize=(8, 5))
ax.axhline(intercept, color="black", ls="--", lw=1, label=r"intercept ($\bar{y}_{train}$)")
for coef_idx in range(max_degree):
    label = rf"$\theta_{{{coef_idx + 1}}}$"
    ax.plot(
        degrees_arr,
        theta_matrix[:, coef_idx],
        marker="o",
        markersize=3,
        label=label,
    )

ax.set_yscale("symlog", linthresh=1)
ax.set_xlabel("Polynomial degree")
ax.set_ylabel(r"Coefficient value $\theta_i$")
ax.set_title(r"OLS coefficients $\theta_i$ vs. polynomial degree")
ax.legend(ncol=2, fontsize="small", loc="upper left", bbox_to_anchor=(1.02, 1.0))
fig.tight_layout()
plt.show()

## Effect of the number of data points

Part (a) also asks us to look at how training-set size affects the degree sweep,
not just the degree itself. We reuse `fit_polynomial_degree_sweep` for a range of
dataset sizes $n \in \{50, 100, 300, 500, 1000\}$ — chosen to span a data-starved
regime ($n=30$, close to the 16 fitted parameters at the highest degree) up to a
comfortably over-determined one ($n=1000$) — and compare the test MSE and $R^2$
curves across degree for each $n$. All runs use the same noise level ($\sigma=0.1$)
and seed, so the only thing that changes between curves is the amount of data.

In [ ]:
# Re-run the same degree sweep for each dataset size, holding noise and seed fixed.
n_values = [50, 100, 300, 500, 1000]
noise_std = 0.1
seed = 42

n_sweep_results = {
    n: fit_polynomial_degree_sweep(
        *generate_runge_data(n=n, noise_std=noise_std, seed=seed), degrees, OLS, seed=seed
    )
    for n in n_values
}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for n in n_values:
    res = n_sweep_results[n]
    axes[0].plot(res["degrees"], res["mse_test"], marker="o", markersize=3, label=f"n={n}")
    axes[1].plot(res["degrees"], res["r2_test"], marker="o", markersize=3, label=f"n={n}")

axes[0].set_yscale("log")
axes[0].set_xlabel("Polynomial degree")
axes[0].set_ylabel("Test MSE")
axes[0].set_title("Test MSE vs. degree, by dataset size")
axes[0].legend(fontsize="small")

axes[1].axhline(0, color="gray", lw=0.5)
axes[1].set_xlabel("Polynomial degree")
axes[1].set_ylabel(r"Test $R^2$")
axes[1].set_title(r"Test $R^2$ vs. degree, by dataset size")
axes[1].legend(fontsize="small")

fig.tight_layout()
plt.show()

**Why does $n=100$ look *worse* than $n=50$ at low/mid degree, with a negative $R^2$?**

This is not a regression bug — the test *MSE* for $n=100$ is actually comparable to
or better than $n=50$ at almost every degree (e.g. degree 4: $0.025$ vs. $0.029$;
degree 10: $0.0094$ vs. $0.0058$). What differs is $R^2$, and that's an artifact of
evaluating on a small, fixed test split: $R^2 = 1 - SS_{res}/SS_{tot}$, and $SS_{tot}$
is just the variance of whatever `y_test` happens to contain.

The Runge function has a sharp peak at $x=0$ and is fairly flat elsewhere. With
`random_state=42`, the $n=50$ test split (10 points) happens to include one point
right next to the peak ($x=-0.067$, $y=0.915$), giving $\mathrm{Var}(y_{test})
\approx 0.056$. The $n=100$ test split (20 points) has no point closer than
$x=-0.258$ to the peak, so $\mathrm{Var}(y_{test}) \approx 0.023$ — less than half.
The same absolute test error therefore produces a much worse, even negative, $R^2$
for $n=100$, even though the underlying fit is not worse.

Takeaway: with a test set this small (10–20 points) and a target with localized
signal, $R^2$ from a single split is noisy and can be misleading — the test MSE
panel above is the more reliable basis for comparing across $n$.

**Why does test MSE get *worse*, not better, as $n$ grows (at low/mid degree)?**

Looking at the full $n$-sweep above, test MSE at low degree actually *increases* with
$n$ — e.g. at degree 1: $0.055$ ($n=50$) $\to$ $0.061$ ($n=100$) $\to$ $0.099$
($n=1000$) — and $n=50$/$n=100$ often beat $n=500$/$n=1000$ even at higher degree. At
first glance that looks backwards: more data should reduce variance and never make an
unregularized OLS fit *worse*. It isn't a modeling problem, though — it's the same
kind of small-test-set artifact as the $R^2$ anomaly discussed above, just visible in
MSE directly because here it's large enough to dominate the trend.

`test_size=0.2` is a fixed *fraction*, so the absolute test-set size scales with $n$:
10 points at $n=50$, 20 at $n=100$, 200 at $n=1000$. The Runge function has a very
narrow, sharp peak around $x=0$ that a low/moderate-degree polynomial systematically
underfits — a *bias* effect, not variance, and it doesn't shrink with $n$. Counting
test points that actually land in that narrow band ($|x| < 0.1$, about 5% of
$[-1, 1]$):

| $n$ | test points | # with $\lvert x \rvert < 0.1$ |
|---|---|---|
| 50 | 10 | 1 |
| 100 | 20 | 0 |
| 1000 | 200 | 21 |

At degree 1, the mean squared error restricted to those near-peak $n=1000$ test
points is $\approx 0.50$, vs. $\approx 0.05$ everywhere else — a $10\times$ gap; by
degree 15, once the model can actually capture the peak, that gap nearly closes
($0.0165$ vs. $0.0122$). The $n=100$ test split contains *zero* such points purely by
chance, so its measured MSE never sees the region where the model is bad and reports
an optimistic number; $n=1000$'s much larger test set reliably samples that region and
reports the true, higher, bias-dominated error. The erratic dip-and-spike in the
$n=50$ curve around degree 10–13 is the same issue in miniature: an MSE computed over
only 10 points is itself a high-variance estimate.

**Takeaway:** larger $n$ here is producing a *more honest* estimate of generalization
error, not a worse model — the small-$n$ curves look better because their tiny,
fixed-fraction test splits are unlikely to sample the one region that's hard to fit.
A single train/test split is fundamentally unreliable for comparing across $n$ when
the target has localized structure like this.

> **TODO (fix once cross-validation is added):** the
> [project task](https://github.com/EducationalMaterialUiO/MachineLearningUiO/blob/main/doc/Projects/2026/Project1/Project1.ipynb)
> asks for $k$-fold cross-validation later in the project. Once that's implemented,
> re-run this $n$-sweep with CV-averaged test MSE (every point acts as a test point
> across folds) instead of one fixed 80/20 split — that should average away this
> fold-composition luck and make the curves monotonically improve with $n$, as
> bias-variance theory predicts. Record the before/after comparison here as evidence
> that the single-split evaluation, not the OLS fit, was the source of this anomaly.

## What does an $n=50$ fit actually look like?

The MSE numbers above don't show *why* the $n=50$ test error can look deceptively
low. Refit at $n=50$ for a few representative degrees, on the same 80/20 split
(`seed=42`) used throughout, and plot each fitted curve against the ground truth —
coloring the 50 sampled points by whether they landed in the train or test split.

Degrees 1, 4, 8, and 15 are chosen to span the full story: degree 1 is far too rigid
to capture the peak at all; degree 4 gets the peak roughly right but already
overshoots near the domain edges; by degree 8 and 15 that edge overshoot has grown
enormous — the classic Runge phenomenon of high-degree polynomial fits. All four
panels share $y$-limits clipped to $[-0.5, 1.3]$ so the curves stay comparable (at
degree 15 the fit actually swings down to $y \approx -28$ just outside the left
edge, far off this scale). Only one of the ten test points — the $\times$ sitting
right at the peak — actually probes the region every low-degree fit gets wrong, so
the aggregate test MSE at $n=50$ barely reflects how bad that fit really is there.

In [ ]:
x50, y50 = generate_runge_data(n=50, noise_std=0.1, seed=42)
x50_train, x50_test, y50_train, y50_test = train_test_split(
    x50, y50, test_size=0.2, random_state=42
)

degrees_to_show = [1, 4, 8, 11, 13, 15]
max_degree_show = max(degrees_to_show)

x_dense = np.linspace(-1, 1, 500)
X_dense_full = univariate_polynomial_design_matrix(
    x=x_dense, degree=max_degree_show, intercept=False
)
X50_train_full = univariate_polynomial_design_matrix(
    x=x50_train, degree=max_degree_show, intercept=False
)
X50_test_full = univariate_polynomial_design_matrix(
    x=x50_test, degree=max_degree_show, intercept=False
)
y50_mean = y50_train.mean()

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True, sharey=True)

for ax, degree in zip(axes.flat, degrees_to_show, strict=True):
    # Same convention as fit_polynomial_degree_sweep: no intercept column, scaler
    # fit on train only, y centered by the training mean.
    X_train_d = X50_train_full[:, :degree]
    X_test_d = X50_test_full[:, :degree]
    X_dense_d = X_dense_full[:, :degree]

    scaler = StandardScaler().fit(X_train_d)
    X_train_s = scaler.transform(X_train_d)
    X_test_s = scaler.transform(X_test_d)
    X_dense_s = scaler.transform(X_dense_d)

    model = OLS().fit(X_train_s, y50_train - y50_mean)
    y_dense_pred = model.predict(X_dense_s) + y50_mean
    y_test_pred = model.predict(X_test_s) + y50_mean
    test_mse = mean_squared_error(y50_test, y_test_pred)

    ax.plot(x_dense, runge_function(x_dense), color="black", lw=1.2, label="Runge's function")
    ax.plot(x_dense, y_dense_pred, color="crimson", lw=1.5, label="OLS fit")
    ax.scatter(x50_train, y50_train, s=20, color="tab:blue", label="train", zorder=3)
    ax.scatter(x50_test, y50_test, s=30, color="tab:orange", marker="x", label="test", zorder=3)
    ax.set_title(f"degree = {degree}, test MSE = {test_mse:.4f}")
    ax.set_ylim(-0.5, 1.3)

axes[0, 0].legend(fontsize="small", loc="upper right")
for ax in axes[-1]:
    ax.set_xlabel("x")
for ax in axes[:, 0]:
    ax.set_ylabel("y")

fig.suptitle(r"OLS fits at $n=50$: underfitting vs. overfitting")
fig.tight_layout()
plt.show()

## The same fits at $n=1000$

Now repeat the exact same plot at $n=1000$, still on an 80/20 split (`seed=42`), so
it sits directly below the $n=50$ version for comparison. This is the other half of
the story from the "why does test MSE get worse as $n$ grows" discussion above: the
underlying fitted curves per degree look essentially the same as at $n=50$ (degree 1
is still a straight line missing the peak; degree 15 still oscillates at the edges)
— what's different is the *test set*. With 200 test points instead of 10, roughly
20 of them now land in the narrow peak region, so the orange $\times$'s densely
trace out the part of the curve every low-degree fit gets wrong. The low-degree
panels below make it visually obvious why their test MSE is higher than the $n=50$
panels above: the test points are actually there to be missed this time.

In [ ]:
x1000, y1000 = generate_runge_data(n=1000, noise_std=0.1, seed=42)
x1000_train, x1000_test, y1000_train, y1000_test = train_test_split(
    x1000, y1000, test_size=0.2, random_state=42
)

X1000_train_full = univariate_polynomial_design_matrix(
    x=x1000_train, degree=max_degree_show, intercept=False
)
X1000_test_full = univariate_polynomial_design_matrix(
    x=x1000_test, degree=max_degree_show, intercept=False
)
y1000_mean = y1000_train.mean()

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True, sharey=True)

for ax, degree in zip(axes.flat, degrees_to_show, strict=True):
    # Same convention as fit_polynomial_degree_sweep: no intercept column, scaler
    # fit on train only, y centered by the training mean.
    X_train_d = X1000_train_full[:, :degree]
    X_test_d = X1000_test_full[:, :degree]
    X_dense_d = X_dense_full[:, :degree]

    scaler = StandardScaler().fit(X_train_d)
    X_train_s = scaler.transform(X_train_d)
    X_test_s = scaler.transform(X_test_d)
    X_dense_s = scaler.transform(X_dense_d)

    model = OLS().fit(X_train_s, y1000_train - y1000_mean)
    y_dense_pred = model.predict(X_dense_s) + y1000_mean
    y_test_pred = model.predict(X_test_s) + y1000_mean
    test_mse = mean_squared_error(y1000_test, y_test_pred)

    ax.plot(x_dense, runge_function(x_dense), color="black", lw=1.2, label="Runge's function")
    ax.plot(x_dense, y_dense_pred, color="crimson", lw=1.5, label="OLS fit")
    ax.scatter(x1000_train, y1000_train, s=8, alpha=0.4, color="tab:blue", label="train", zorder=3)
    ax.scatter(
        x1000_test,
        y1000_test,
        s=14,
        alpha=0.6,
        color="tab:orange",
        marker="x",
        label="test",
        zorder=3,
    )
    ax.set_title(f"degree = {degree}, test MSE = {test_mse:.4f}")
    ax.set_ylim(-0.5, 1.3)

axes[0, 0].legend(fontsize="small", loc="upper right")
for ax in axes[-1]:
    ax.set_xlabel("x")
for ax in axes[:, 0]:
    ax.set_ylabel("y")

fig.suptitle(r"OLS fits at $n=1000$: same underfitting/overfitting, denser test coverage")
fig.tight_layout()
plt.show()